# Lab 10 Part 1: Retrieval-Augmented Generation (RAG)

## Introduction

In this lab, we will explore Retrieval-Augmented Generation (RAG), a method that enhances Large Language Models (LLMs) by combining them with a retrieval mechanism. RAG retrieves relevant documents from an external knowledge base to provide context for answering user queries. This approach mitigates the issues of hallucination and outdated knowledge by grounding responses in retrieved documents.

### How RAG Works

1. **Document Encoder**: Converts documents into dense vector embeddings using sentence transformers, enabling efficient semantic similarity search.
2. **Retriever**: Matches user queries to the most relevant documents based on cosine similarity between embeddings.
3. **Generator (LLM)**: Uses the retrieved documents to answer queries, strictly adhering to the provided context.

By the end of this lab, you will:
- Use sentence transformers as document encoders (all-MiniLM-L6-v2 or all-mpnet-base-v2).
- Experiment with different LLMs: Llama 3.2 3B, Gemma 3 4B, Qwen 2.5 3B, or Phi-3 Mini.
- Evaluate model performance using semantic similarity metrics.
- Design prompts that prevent LLMs from hallucinating.
- Implement a pipeline to work with PDFs for RAG (optional).

---

## Exercise 1: Document Encoding and Query Generation

### Objective

Set up a RAG pipeline using a sentence transformer as the document encoder and an LLM to generate answers. Design effective prompts that prevent hallucination.

### Instructions

1. Use the following tools and models:
   - **Document Encoder**: Sentence transformer (all-MiniLM-L6-v2 or all-mpnet-base-v2).
   - **Generators**: Choose from Llama 3.2 3B, Gemma 3 4B, Qwen 2.5 3B, or Phi-3 Mini.

2. Implement a pipeline where:
   - Documents are encoded into embeddings during setup.
   - User queries are matched with relevant documents using cosine similarity.
   - A similarity threshold filters out irrelevant documents.
   - Retrieved context and query are formatted into a structured prompt.
   - The LLM generates responses based only on the retrieved context.

3. Design a prompt that:
   - Explicitly forbids the model from using its own knowledge.
   - Provides clear instructions to answer only from context.
   - Includes a mandatory fallback: "I'm sorry, I cannot answer this question based on the provided context."
   - Uses the model's specific chat template format.

4. Experiment with generation parameters:
   - `top_k`: Number of documents to retrieve (1, 3, 5, 10)
   - `similarity_threshold`: Minimum cosine similarity (0.3, 0.5, 0.7)
   - `num_beams`: Beam search width (1, 3, 5)
   - `temperature`: Sampling temperature (0.3, 0.7, 1.2)
   - `do_sample`: Deterministic vs stochastic sampling
   - `max_new_tokens`: Maximum response length (50, 100, 150)

### Tasks

- Test different parameter values and observe effects on response quality.
- Strengthen the system prompt to prevent hallucination.
- Compare different LLMs for response quality, speed, and memory usage.

---

## Exercise 2: Evaluating RAG with Semantic Similarity

### Objective

Evaluate the quality of LLM responses using semantic similarity scores between generated answers and reference answers.

### Instructions

1. Use the provided evaluation dataset containing:
   - A knowledge base of 20 diverse documents.
   - 13 test queries with reference answers.
   - Mix of answerable and unanswerable questions.

2. For each query:
   - Retrieve relevant documents using semantic search.
   - Generate an answer using the LLM.
   - Calculate semantic similarity between generated and reference answers using the sentence transformer.
   - Similarity scores range from 0.0 (completely different) to 1.0 (identical meaning).

3. Run evaluation using `evaluate_queries()`:
   - Test different parameter combinations.
   - Record average semantic similarity scores (target: >0.6 for good performance).

### Tasks

- Run baseline evaluation with default parameters (top_k=3, temp=0.7, num_beams=3).
- Test 6 parameter combinations: greedy decoding, high beam search, low temperature, high temperature, more context, stricter threshold.
- Create a comparison table for different models: Average Similarity, Inference Time, Memory Usage.
- Examine bottom-performing queries and identify failure causes: poor retrieval, weak generation, or insufficient context.
- Reflect on trade-offs between model size, performance, and speed.

---

## Exercise 3 (Extra): PDF Knowledge Base for RAG

### Objective

Extend the RAG pipeline to work with PDF documents for real-world question-answering.

### Instructions

1. Install PDF processing library: pypdf2 or pdfplumber.

2. Implement preprocessing:
   - Extract text from PDF files in a folder.
   - Split text into chunks (sentence-based or fixed-size with overlap).
   - Store metadata: filename, page number, chunk ID.
   - Encode chunks using the sentence transformer.
   - Save embeddings, chunks, and metadata to disk.

3. At inference:
   - Load precomputed embeddings from disk.
   - Match queries with PDF chunk embeddings.
   - Retrieve relevant chunks as context for the LLM.
   - Optionally return source PDF and page number with answers.

4. Test the pipeline:
   - Prepare a folder with 3-5 PDF documents.
   - Ask factual, comparative, summary, and unanswerable questions.

### Tasks

- Implement text extraction and chunking functions.
- Experiment with chunk sizes (100, 200, 500 words) and overlap strategies.
- Create 10-15 test questions based on PDF content and evaluate using semantic similarity.
- Calculate retrieval accuracy: percentage of queries where correct chunk is in top-k.
- Reflect on PDF extraction challenges and scalability to thousands of documents.

---

## Tips and Best Practices

### Prompt Engineering
- Be explicit: "Use ONLY the context" prevents hallucination.
- Always provide a fallback response for unanswerable questions.
- Test edge cases: ambiguous queries, no relevant context.

### Performance Optimization
- Precompute and save document embeddings to disk.
- Use FP16 for 2x speed and 50% memory reduction.
- Adjust max_new_tokens to avoid over-generation.
- Clear GPU cache after large batches.

### Evaluation
- Semantic similarity scores range from 0.0 to 1.0; target >0.6 for good performance.
- Also evaluate factual accuracy and relevance manually.
- Analyze failures to distinguish retrieval vs generation issues.

### Troubleshooting
- **Model uses own knowledge**: Strengthen system prompt and lower temperature.
- **Low similarity scores**: Check retrieval quality, adjust threshold and top_k.
- **Out of memory**: Reduce batch_size, max_new_tokens, or use smaller model.

---
Good luck with your experiments! 🎓

In [ ]:
# ============================================================================
# SECTION 1: INSTALLATION
# ============================================================================

# Install PyTorch with GPU support (CUDA 12.1 for modern GPUs like T4, A100, V100)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core libraries with specific versions for reproducibility
!pip install transformers         # Hugging Face transformers for LLMs
!pip install sentence-transformers  # For encoding text into embeddings
!pip install nltk                   # Natural Language Toolkit for BLEU scores
!pip install numpy               # Numerical operations
!pip install accelerate           # Efficient model loading and memory management
!pip install pypdf

# Download required NLTK data for tokenization and BLEU score calculation
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# ============================================================================
# SECTION 2: BASIC RAG PIPELINE
# ============================================================================

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings

# --- Setup ---
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Load Models ---
print("Loading models...")
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
model_name = "unsloth/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token
generator = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)

# --- Knowledge Base & Encoding ---
knowledge_base = [
    "LLaMA is a large language model developed by Meta.",
    "Sentence Transformers are used for dense vector similarity.",
    "RAG is a technique that enhances the accuracy and reliability of AI models by retrieving relevant information from an external knowledge.",
    "PyTorch is a popular framework for deep learning.",
    "The Eiffel Tower is located in Paris, France."
]
kb_embeddings = retriever.encode(knowledge_base, convert_to_tensor=True, normalize_embeddings=True)

# --- RAG Generation Function ---
def rag_generate(query, top_k=3, similarity_threshold=0.3, temperature=0.7, do_sample=False, num_beams=3):
    # 1. Retrieve
    query_emb = retriever.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_emb, kb_embeddings)[0]
    top_results = torch.topk(scores, k=min(top_k, len(knowledge_base)))

    # 2. Filter
    context_docs = []
    for idx, score in zip(top_results.indices, top_results.values):
        if score >= similarity_threshold:
            context_docs.append(knowledge_base[idx])

    if not context_docs:
        return "I'm sorry, I cannot answer this question based on the provided context."

    context_text = "\n".join(context_docs)

    # 3. Prompt Engineering (Strict System Prompt)
    system_prompt = """You are a strict question-answering assistant.
CRITICAL:
1. Use ONLY the provided Context.
2. If the Context does not contain the answer, respond EXACTLY with: "I'm sorry, I cannot answer this question based on the provided context."
3. DO NOT use outside knowledge."""

    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n{context_text}\n\nQuestion: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"

    # 4. Generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=100,
            temperature=temperature,
            do_sample=do_sample,
            num_beams=num_beams,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1
        )

    return tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# --- Experiments ---
print("\n--- Experiment 1: Standard Query (Beam Search) ---")
print(rag_generate("What is RAG?", top_k=3, num_beams=3))

print("\n--- Experiment 2: Hallucination Check (Unknown Topic) ---")
print(rag_generate("Who discovered Penicillin?", top_k=3))

print("\n--- Experiment 3: Greedy Decoding (Faster) ---")
print(rag_generate("Where is the Eiffel Tower?", do_sample=False, num_beams=1))

print("\n--- Experiment 4: High Creativity (High Temp) ---")
print(rag_generate("What is PyTorch?", temperature=1.2, do_sample=True))

In [ ]:
# ============================================================================
# SECTION 3: EVALUATION WITH SEMANTIC SIMILARITY
# ============================================================================

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
import time

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load Models & Data ---
retriever = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
generator = AutoModelForCausalLM.from_pretrained("unsloth/Llama-3.2-3B-Instruct", torch_dtype=torch.float16, device_map="auto")

knowledge_base = [
    "Large Language Models (LLMs) like Llama 3 are trained using the Transformer architecture, which relies on self-attention mechanisms.",
    "The Attention Mechanism allows a model to weigh the importance of different words in a sequence, regardless of their distance.",
    "Supervised Learning is a type of machine learning where the model is trained on labeled data.",
    "Unsupervised Learning involves training a model on data without explicit labels to find hidden patterns or structures.",
    "Reinforcement Learning (RL) is an area of ML concerned with how intelligent agents take actions in an environment to maximize cumulative reward.",
    "Convolutional Neural Networks (CNNs) are primarily used for image recognition and processing due to their ability to capture spatial hierarchies.",
    "Recurrent Neural Networks (RNNs) are designed for sequential data, though they often struggle with long-term dependencies compared to Transformers.",
    "Backpropagation is the primary algorithm used to train neural networks by calculating the gradient of the loss function with respect to the weights.",
    "Overfitting occurs when a model learns the training data too well, including the noise, leading to poor performance on unseen data.",
    "Underfitting happens when a model is too simple to capture the underlying trend of the data.",
    "Gradient Descent is an optimization algorithm used to minimize the loss function by iteratively moving in the direction of the steepest descent.",
    "A Vector Database stores data as high-dimensional embeddings, allowing for semantic search rather than just keyword matching.",
    "Tokenization is the process of breaking down text into smaller units like words or sub-words for a model to process.",
    "The 'Cold Start' problem in recommendation systems refers to the difficulty in making recommendations for new users or items with no history.",
    "Turing Completeness refers to a system's ability to simulate any Turing machine, meaning it can perform any computation given enough time and memory.",
    "The CPU (Central Processing Unit) is the primary component of a computer that performs most of the processing, often called the 'brain'.",
    "A GPU (Graphics Processing Unit) is specialized for parallel processing, making it highly efficient for training deep learning models.",
    "TPUs (Tensor Processing Units) are AI-accelerator application-specific integrated circuits (ASICs) developed by Google specifically for machine learning.",
    "Regularization techniques like L1 and L2 are used to prevent overfitting by adding a penalty to the loss function based on the size of the weights.",
    "The Vanishing Gradient Problem occurs when gradients become so small during backpropagation that the weights stop updating effectively.",
    "Fine-tuning is the process of taking a pre-trained model and training it further on a specific, smaller dataset to adapt it to a particular task.",
    "Data Augmentation is a technique used to increase the diversity of training data without collecting new data, often by applying transformations like rotation or cropping.",
    "In binary classification, Precision is the ratio of correctly predicted positive observations to the total predicted positives.",
    "Recall (or Sensitivity) is the ratio of correctly predicted positive observations to all actual observations in the class.",
    "The F1 Score is the harmonic mean of Precision and Recall, providing a balance between the two metrics."
]

kb_embeddings = retriever.encode(knowledge_base, convert_to_tensor=True, normalize_embeddings=True)

test_queries = [
    ("How do Transformers process word importance?", "The Attention Mechanism allows a model to weigh the importance of different words in a sequence, regardless of their distance."),
    ("What is the difference between supervised and unsupervised learning?", "Supervised Learning uses labeled data, while Unsupervised Learning finds patterns in data without explicit labels."),
    ("Which neural network is best for images?", "Convolutional Neural Networks (CNNs) are primarily used for image recognition and processing."),
    ("What is the goal of Reinforcement Learning?", "Reinforcement Learning is concerned with how agents take actions to maximize cumulative reward."),
    ("How does backpropagation work?", "It is the primary algorithm used to train neural networks by calculating the gradient of the loss function with respect to the weights."),
    ("What happens during overfitting?", "The model learns the training data and its noise too well, leading to poor performance on new data."),
    ("Define Gradient Descent.", "An optimization algorithm used to minimize the loss function by iteratively moving in the direction of the steepest descent."),
    ("What is a Vector Database used for?", "It stores data as high-dimensional embeddings, allowing for semantic search."),
    ("What are TPUs?", "TPUs are AI-accelerator application-specific integrated circuits developed by Google for machine learning."),
    ("Why are GPUs used in ML?", "A GPU is specialized for parallel processing, making it highly efficient for training deep learning models."),
    ("What is the Vanishing Gradient Problem?", "It occurs when gradients become so small during backpropagation that weights stop updating effectively."),
    ("How do you prevent overfitting using math?", "Regularization techniques like L1 and L2 add a penalty to the loss function based on weight size."),
    ("What is the F1 Score?", "The F1 Score is the harmonic mean of Precision and Recall."),
    ("What is tokenization?", "The process of breaking down text into smaller units like words or sub-words."),
    ("What is the brain of the computer?", "The CPU (Central Processing Unit) is the primary component that performs most of the processing."),
    ("Define Turing Completeness.", "A system's ability to simulate any Turing machine and perform any computation."),
    ("What is the cold start problem?", "The difficulty in making recommendations for new users or items with no history."),
    ("What is fine-tuning?", "The process of taking a pre-trained model and training it further on a specific, smaller dataset."),
    ("How does data augmentation work?", "It increases training data diversity without new data by applying transformations like rotation."),
    ("What is recall in classification?", "The ratio of correctly predicted positive observations to all actual observations in the class."),
    # Unanswerable / Out-of-Scope
    ("Who founded Microsoft?", "I'm sorry, I cannot answer this question based on the provided context."),
    ("What is the capital of Iceland?", "I'm sorry, I cannot answer this question based on the provided context."),
    ("How do you bake a cake?", "I'm sorry, I cannot answer this question based on the provided context."),
    ("What is the price of Bitcoin?", "I'm sorry, I cannot answer this question based on the provided context."),
    ("Who won the 2024 Super Bowl?", "I'm sorry, I cannot answer this question based on the provided context.")
]

# --- RAG Generation Function ---
def rag_generate_eval(query, top_k=3, similarity_threshold=0.3, temperature=0.7, do_sample=False, num_beams=3):
    query_emb = retriever.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_emb, kb_embeddings)[0]
    top_results = torch.topk(scores, k=min(top_k, len(knowledge_base)))
    
    context_docs = []
    for idx, score in zip(top_results.indices, top_results.values):
        if score >= similarity_threshold:
            context_docs.append(knowledge_base[idx])
    
    if not context_docs:
        return "I'm sorry, I cannot answer this question based on the provided context."
    
    context_text = "\n".join(context_docs)
    system_prompt = """You are a strict question-answering assistant.
CRITICAL:
1. Use ONLY the provided Context.
2. If the Context does not contain the answer, respond EXACTLY with: "I'm sorry, I cannot answer this question based on the provided context."
3. DO NOT use outside knowledge."""
    
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n{context_text}\n\nQuestion: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    
    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=100,
            temperature=temperature,
            do_sample=do_sample,
            num_beams=num_beams,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1
        )
    
    return tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# --- Evaluation Function ---
def evaluate_queries(queries, **gen_params):
    results = []
    total_similarity = 0
    start_time = time.time()
    
    for query, reference in queries:
        generated = rag_generate_eval(query, **gen_params)
        
        # Calculate semantic similarity
        gen_emb = retriever.encode(generated, convert_to_tensor=True, normalize_embeddings=True)
        ref_emb = retriever.encode(reference, convert_to_tensor=True, normalize_embeddings=True)
        similarity = util.cos_sim(gen_emb, ref_emb).item()
        
        total_similarity += similarity
        results.append({
            'query': query,
            'reference': reference,
            'generated': generated,
            'similarity': similarity
        })
    
    elapsed_time = time.time() - start_time
    avg_similarity = total_similarity / len(queries)
    
    return results, avg_similarity, elapsed_time

# --- Run Baseline Evaluation ---
print("\n=" * 80)
print("BASELINE EVALUATION")
print("=" * 80)
results, avg_sim, time_taken = evaluate_queries(test_queries, top_k=3, temperature=0.7, num_beams=3)
print(f"\nAverage Similarity: {avg_sim:.4f}")
print(f"Time Taken: {time_taken:.2f}s")
print(f"\nTop 5 Results:")
for i, r in enumerate(results[:5]):
    print(f"\n{i+1}. Query: {r['query']}")
    print(f"   Generated: {r['generated']}")
    print(f"   Similarity: {r['similarity']:.4f}")

# --- Parameter Experiments ---
print("\n" + "=" * 80)
print("PARAMETER EXPERIMENTS")
print("=" * 80)

experiments = [
    {"name": "Greedy Decoding", "params": {"top_k": 3, "temperature": 0.7, "num_beams": 1, "do_sample": False}},
    {"name": "High Beam Search", "params": {"top_k": 3, "temperature": 0.7, "num_beams": 5, "do_sample": False}},
    {"name": "Low Temperature", "params": {"top_k": 3, "temperature": 0.3, "num_beams": 3, "do_sample": True}},
    {"name": "High Temperature", "params": {"top_k": 3, "temperature": 1.2, "num_beams": 3, "do_sample": True}},
    {"name": "More Context", "params": {"top_k": 5, "temperature": 0.7, "num_beams": 3, "do_sample": False}},
    {"name": "Stricter Threshold", "params": {"top_k": 3, "similarity_threshold": 0.5, "temperature": 0.7, "num_beams": 3}}
]

for exp in experiments:
    print(f"\n{exp['name']}:")
    _, avg_sim, time_taken = evaluate_queries(test_queries[:10], **exp['params'])
    print(f"  Avg Similarity: {avg_sim:.4f}  |  Time: {time_taken:.2f}s")

# --- Bottom Performers Analysis ---
print("\n" + "=" * 80)
print("BOTTOM 5 PERFORMERS")
print("=" * 80)
sorted_results = sorted(results, key=lambda x: x['similarity'])
for i, r in enumerate(sorted_results[:5]):
    print(f"\n{i+1}. Query: {r['query']}")
    print(f"   Reference: {r['reference']}")
    print(f"   Generated: {r['generated']}")
    print(f"   Similarity: {r['similarity']:.4f}")

In [ ]:
# ============================================================================
# SECTION 4: PDF RAG (OPTIONAL/EXTRA)
# ============================================================================

from pypdf import PdfReader
import os
import re

# --- PDF Processing Functions ---
def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file."""
    reader = PdfReader(pdf_path)
    text = ""
    for page_num, page in enumerate(reader.pages):
        text += f"\n[Page {page_num + 1}]\n" + page.extract_text()
    return text

def chunk_text(text, chunk_size=200, overlap=50):
    """Split text into chunks with overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

def process_pdf_folder(folder_path, chunk_size=200):
    """Process all PDFs in a folder and create knowledge base."""
    pdf_kb = []
    pdf_metadata = []
    
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            text = extract_text_from_pdf(pdf_path)
            chunks = chunk_text(text, chunk_size=chunk_size)
            
            for i, chunk in enumerate(chunks):
                pdf_kb.append(chunk)
                pdf_metadata.append({
                    "filename": filename,
                    "chunk_id": i,
                    "source": f"{filename} (Chunk {i})"
                })
    
    return pdf_kb, pdf_metadata

# --- PDF RAG Pipeline ---
def pdf_rag_generate(query, pdf_kb, pdf_embeddings, top_k=3, similarity_threshold=0.3):
    """Generate answer from PDF knowledge base."""
    query_emb = retriever.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_emb, pdf_embeddings)[0]
    top_results = torch.topk(scores, k=min(top_k, len(pdf_kb)))
    
    context_docs = []
    for idx, score in zip(top_results.indices, top_results.values):
        if score >= similarity_threshold:
            context_docs.append(pdf_kb[idx])
    
    if not context_docs:
        return "I'm sorry, I cannot answer this question based on the provided context."
    
    context_text = "\n\n".join(context_docs)
    system_prompt = """You are a strict question-answering assistant.
CRITICAL:
1. Use ONLY the provided Context.
2. If the Context does not contain the answer, respond EXACTLY with: "I'm sorry, I cannot answer this question based on the provided context."
3. DO NOT use outside knowledge."""
    
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n{context_text}\n\nQuestion: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    
    with torch.no_grad():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            num_beams=3,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1
        )
    
    return tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# --- Example Usage (Uncomment and modify path to use) ---
# pdf_folder = "./pdfs"  # Replace with your PDF folder path
# pdf_kb, pdf_metadata = process_pdf_folder(pdf_folder, chunk_size=200)
# pdf_embeddings = retriever.encode(pdf_kb, convert_to_tensor=True, normalize_embeddings=True)
# 
# # Test query
# query = "What is the main topic discussed in the document?"
# answer = pdf_rag_generate(query, pdf_kb, pdf_embeddings, top_k=3)
# print(f"Query: {query}")
# print(f"Answer: {answer}")

print("PDF RAG functions defined. Uncomment the example usage section to test with your PDFs.")